# Research 03 — PPO/GRPO loss：clip、advantage、KL

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Papa-Panda/field-notes/blob/main/pytorch/research/03-ppo-grpo-loss.ipynb)

- **日期**：2026-10-02
- **定位**：Research PyTorch 轮次的 RL 算法面。DPO 是离线偏好优化，PPO/GRPO 是在线 RL —— 这一轮要能手写任意一个 loss 并解释每一项的来历。
- **环境**：纯 PyTorch，CPU 即可。

配套笔记：[03-ppo-grpo-loss.md](03-ppo-grpo-loss.md)


## 今日目标

1. 手写 PPO 的 clipped loss，并数值验证 clip 的"可信域外梯度置零"行为。
2. 手写 GAE，验证 $ \lambda=0 $ 和 $ \lambda=1 $ 两个极端。
3. 手写 GRPO 的组内 advantage + KL，用 toy bandit 训出可见的策略改进。

记号：$ \rho_t = \pi_\theta(a_t|s_t) / \pi_{old}(a_t|s_t) $ ，$ A_t $ advantage，$ \epsilon $ clip 幅度（常用 0.2）。


## 1. PPO clipped loss

$$ \mathcal{L}^{clip} = \mathbb{E}\left[\min\left(\rho_t A_t, \mathrm{clip}(\rho_t, 1-\epsilon, 1+\epsilon) A_t\right)\right] $$

$ \min $ 取悲观估计：$ A>0 $ 时防止一步冲太远，$ A<0 $ 时防止把坏动作的概率降过头。$ \rho $ 跑出区间后，那一项的梯度为零 —— 不是把 ratio 拉回来，是拒绝在可信域外优化。


In [ ]:
import torch

def ppo_loss(logp_new, logp_old, advantages, clip_eps=0.2):
    """PPO 的 clipped surrogate loss。

    入参（三个张量形状必须一致，(B,) 序列级或 (B, T) token 级）:
      logp_new: 当前策略 πθ 对已采样动作的 log 概率（带梯度）
      logp_old: 采样时的行为策略对同一批动作的 log 概率（不带梯度，当常数）
      advantages: 同形状的 advantage（已 detach，梯度只经 logp_new 回传）
      clip_eps: trust region 半宽（默认 0.2，ratio 被夹在 [0.8, 1.2]）
    返回: 标量 loss（最大化目标取负，交给 optimizer 最小化）。"""
    ratio = torch.exp(logp_new - logp_old)                    # ← log 空间相减后 exp
    surr1 = ratio * advantages
    surr2 = torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps) * advantages
    return -torch.min(surr1, surr2).mean()                    # ← 最大化目标 → 取负

# 检查 1：policy 未动（ratio=1）时 loss 应等于 -mean(A)
A = torch.tensor([1.0, -1.0, 2.0])
lp = torch.tensor([-0.5, -1.5, -2.0])
print('检查 1: ratio=1 时 loss =', round(ppo_loss(lp, lp, A).item(), 4),
      ' 期望 -mean(A) =', round(-A.mean().item(), 4))

# 检查 2：A>0 且 ratio 冲出上界 → 该项梯度应为 0（可信域外拒绝优化）
lp_new = torch.tensor([-0.5 + 0.9], requires_grad=True)   # ratio = e^0.9 ≈ 2.46 >> 1.2
ppo_loss(lp_new, torch.tensor([-0.5]), torch.tensor([1.0]), 0.2).backward()
print('检查 2: 冲出 clip 上界后梯度 =', lp_new.grad.item(), '（应为 0）')

# 检查 3：A<0 且 ratio 已在下界内侧 → min 取悲观项，梯度正常回传
lp_new = torch.tensor([-0.5], requires_grad=True)          # ratio = 1
ppo_loss(lp_new, torch.tensor([-0.5]), torch.tensor([-1.0]), 0.2).backward()
print('检查 3: 区间内梯度 =', round(lp_new.grad.item(), 3), '（非零，正常回传）')


## 2. GAE：advantage 的估计器

TD 残差 $ \delta_t = r_t + \gamma V(s_{t+1}) - V(s_t) $ ，$ A^{GAE}_t = \sum_l (\gamma\lambda)^l \delta_{t+l} $ 。反向递推实现。


In [ ]:
def compute_gae(rewards, values, gamma=0.99, lam=0.95):
    """GAE(λ)：把 TD 残差 δ_t 按 (γλ)^k 衰减加权求和。

    rewards: (T,) 每步的 reward；
    values: (T+1,) critic 的 V(s_t) —— 比 rewards 多一位，最后一位是 bootstrap
            的 V(s_T)（序列被截断时用它续命，自然终止填 0）；
    gamma: 折扣 γ；lam: λ，0 = 纯 TD、1 = 蒙特卡洛（下面两个极端自检各验一头）。
    返回: (T,) 每一步的 advantage A_t。"""
    adv = torch.zeros_like(rewards)
    gae = 0.0
    for t in reversed(range(len(rewards))):
        delta = rewards[t] + gamma * values[t + 1] - values[t]
        gae = delta + gamma * lam * gae
        adv[t] = gae
    return adv

r = torch.tensor([1.0, 0.0, 0.0, 1.0])
v0 = torch.zeros(5)
print('λ=0（纯 TD）:', [round(x, 2) for x in compute_gae(r, v0, 1.0, 0.0).tolist()],
      ' 期望=δ 本身:', [1.0, 0.0, 0.0, 1.0])
print('λ=1, γ=1, V=0（蒙特卡洛）:', [round(x, 2) for x in compute_gae(r, v0, 1.0, 1.0).tolist()],
      ' 期望=未来 reward 之和:', [2.0, 1.0, 1.0, 1.0])


## 3. GRPO：组内相对 advantage

同一 prompt 采样 $ G $ 个回答，$ A_i = \frac{R_i - \mathrm{mean}(R_{1..G})}{\mathrm{std}(R_{1..G})} $ 。组内 advantage 之和恒为 0 —— 梯度只推"比同组平均好"的样本。下面用 toy bandit 把整个闭环跑一遍：策略是 6 个动作上的 logits，目标是学会选动作 3。


In [ ]:
import torch.nn.functional as F

def grpo_advantages(rewards):
    """GRPO 的组内标准化 advantage（用同组样本当基线，省掉 critic）。

    rewards: (P, G) —— P 个 prompt，每个 prompt 采样 G 个 completion 得到的 reward。
    返回: (P, G)，每组内 (r − mean) / std。同一 prompt 的样本互为基线。"""
    mean = rewards.mean(dim=1, keepdim=True)
    std = rewards.std(dim=1, keepdim=True)
    return (rewards - mean) / (std + 1e-8)     # Dr. GRPO 主张去掉 std 项防难度偏置

torch.manual_seed(0)
P, G, K, TARGET = 16, 8, 6, 3
logits = torch.zeros(K, requires_grad=True)          # 策略：动作上的 logits
ref_logp = torch.log_softmax(torch.zeros(K), -1)     # ref = 均匀分布
opt = torch.optim.Adam([logits], lr=0.1)

for step in range(200):
    logp = torch.log_softmax(logits, -1)
    probs = logp.exp()
    actions = torch.multinomial(probs, P * G, replacement=True).view(P, G)
    rewards = (actions == TARGET).float()
    adv = grpo_advantages(rewards)
    assert adv.sum().abs() < 1e-4, '组内 advantage 之和应为 0'

    # 策略梯度项：-logp(a) * A；KL 项（对 ref 的解析 KL，拴住别跑偏）
    pg = -(logp[actions] * adv).mean()
    kl = (probs * (logp - ref_logp)).sum()
    loss = pg + 0.01 * kl
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 40 == 0:
        print(f'step {step:3d}  P(动作3) {probs[TARGET].item():.3f}  loss {loss.item():.4f}')
print('\n最终动作分布:', [round(x, 3) for x in probs.tolist()], '（目标动作是 3）')


## 4. KL 估计器三兄弟（实现里最容易写错符号的地方）

采样 $ a \sim \pi_\theta $ ，记 $ r = \pi_{ref}(a)/\pi_\theta(a) $ ：

- $ k_1 = -\log r = \log\pi_\theta - \log\pi_{ref} $ ：无偏、高方差、单样本可为负
- $ k_2 = \frac{1}{2}(\log\pi_\theta - \log\pi_{ref})^2 $ ：低方差、有偏
- $ k_3 = (r - 1) - \log r $ ：无偏、恒非负，实现里最常见


In [ ]:
def kl_estimators(logp, ref_logp):
    """三种 KL 估计量，输入是同一批采样 token 上的 log 概率。

    logp: (N,) 样本在当前策略 πθ 下的 log p（样本本身从 πθ 采的）；
    ref_logp: (N,) 同一采样在 ref 模型下的 log p。
    返回: (k1, k2, k3)，各 (N,)；对 batch 取均值即 KL(πθ || πref) 的估计。"""
    r = torch.exp(ref_logp - logp)
    k1 = logp - ref_logp
    k2 = 0.5 * (logp - ref_logp) ** 2
    k3 = (r - 1) - (ref_logp - logp)
    return k1, k2, k3

# 数值自检：从 π_θ 采样，三个估计器的均值都应逼近 KL(π_θ || π_ref)
torch.manual_seed(0)
logits = torch.tensor([2.0, 1.0, 0.0, -1.0])
ref_logits = torch.zeros(4)
lp, rlp = torch.log_softmax(logits, -1), torch.log_softmax(ref_logits, -1)
true_kl = (lp.exp() * (lp - rlp)).sum()
samples = torch.multinomial(lp.exp(), 20000, replacement=True)
k1, k2, k3 = kl_estimators(lp[samples], rlp[samples])
print(f'真实 KL = {true_kl.item():.4f} | k1 均值 {k1.mean().item():.4f}（可为负: {(k1<0).float().mean().item():.2f}）'
      f' | k2 均值 {k2.mean().item():.4f} | k3 均值 {k3.mean().item():.4f}（恒非负: {(k3>=0).all().item()}）')


## 5. 三方对照 + debug 检查清单

| | DPO | PPO | GRPO |
|---|---|---|---|
| 数据 | 离线偏好对 | 在线采样 | 在线采样（组内） |
| 网络 | policy + ref | policy/ref/reward/critic | policy + ref |
| advantage | 隐式（log-ratio 差） | GAE + critic | 组内标准化 |
| KL 位置 | 闭式解里 | reward 里或 loss 里 | loss 里显式项 |

**现场 debug 顺序（每一步自带修复）**：

1. **ratio 分布不对** —— logp 差忘了做 exp 就当 ratio 用了。
   **修复**：ratio = exp(logp_new − logp_old)；先打印分布自检，新旧同策略时应恒为 1（第 2 节的数值检查就是这把尺）。
2. **prompt token 漏进 loss** —— mask 没切干净，模型在学背 prompt。
   **修复**：loss 只对 completion token 求和，mask 在 `get_batch_logps` 里就处理掉（Research 02 同款）。
3. **advantage 广播维度错** —— 序列级 advantage 摊到 token 级时维度没对齐，广播出怪形状也不报错。
   **修复**：广播前确认 advantage 与 logp 的形状关系，序列级标量先 unsqueeze 到 token 维。
4. **KL 符号反** —— KL 项本该是惩罚，写反了变成奖励发散。
   **修复**：符号进 loss 前，先用第 4 节的 k1/k2/k3 估计量对拍真值 KL，估出来是负的先停下查符号。
5. **clip 区间方向反** —— `clip(ratio, 1+ε, 1−ε)` 上下界写颠倒，trust region 形同虚设。
   **修复**：上下界别写反；用第 2 节的检查验一遍（trust region 外梯度应为 0，反了立刻现形）。

## 小结

- PPO 的命门：ratio 的 log 空间计算、clip 的 min 悲观语义、可信域外梯度置零。
- GAE 是"用 V 把未来折现回来"，两个极端 λ=0/λ=1 必须能脱口而出。
- GRPO 用组内名次换掉 critic：组内 advantage 和为 0、std 位置（组内不是全体）是最容易错的两点。
- KL 符号方向写反是静默杀手：训完离 ref 越来越远，loss 还在降。

**下一篇**：Research 04 — 注意力机制手写（SDPA、多头、causal mask 与 KV cache）
